# SW-14 — Le coup ontologique comme diff de graphe exécutable

[Version Python](SW-14-Coup-Ontologique-Python.ipynb) · [Sommaire SemanticWeb](README.md) · [SW-13 : raisonneurs](SW-13-Reasoners-CSharp.ipynb)

**Objectifs** : construire un diff RDF ; valider une extension par SHACL ; comparer des fermetures OWL-RL et tracer une assertion par réification. **Prérequis** : RDF, SPARQL, OWL et SHACL (SW-7, SW-8, SW-13). **Durée** : 90 minutes.

Le graphe et les requêtes utilisent dotNetRDF ; son moteur SHACL valide les données. Pour la fermeture OWL-RL, Python.NET appelle le même moteur `owlrl` que le carnet Python : un raisonneur RDFS seul ne couvre pas OWL-RL. Versions vérifiées : CPython 3.13.16, dotNetRDF 3.5.2, Python.NET 3.0.5, owlrl 7.1.4 et rdflib 7.6.0.

### Préparation sous Linux

Avec CPython 3.13 disposant de sa bibliothèque partagée, Jupyter et le kernel .NET Interactive déjà installés, lancez depuis un terminal :

```bash
python3.13 -m venv .venv-sw14
source .venv-sw14/bin/activate
python -m pip install 'owlrl==7.1.4' 'rdflib==7.6.0'
export PYTHONNET_PYDLL="$(python -c 'import os, sysconfig; print(os.path.join(sysconfig.get_config_var("LIBDIR") or "", sysconfig.get_config_var("LDLIBRARY") or ""))')"
export OWL_RL_SITE_PACKAGES="$(python -c 'import sysconfig; print(sysconfig.get_paths()["purelib"])')"
if [ -f "$PYTHONNET_PYDLL" ] && [ -d "$OWL_RL_SITE_PACKAGES" ]; then
    python -c 'import rdflib, owlrl; print("rdflib", rdflib.__version__, "owlrl", owlrl.__version__)'
    jupyter lab
else
    echo "Bibliothèque CPython ou site-packages introuvable : vérifier l'installation Python."
fi
```

Démarrez Jupyter après ces exports : le kernel hérite des chemins. La bibliothèque partagée peut appartenir à l'installation Python de base plutôt qu'au venv. Sous Windows ou macOS, le nom et l'emplacement de la bibliothèque diffèrent ; adaptez l'activation du venv et `PYTHONNET_PYDLL` à votre installation.

In [1]:
#r "nuget: dotNetRDF, 3.5.2"
#r "nuget: pythonnet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using VDS.RDF;
using VDS.RDF.Parsing;
using VDS.RDF.Writing;
using VDS.RDF.Query;
using VDS.RDF.Shacl;
using Python.Runtime;
Console.WriteLine("dotNetRDF et Python.NET chargés ; RDF, SPARQL et SHACL disponibles.");

Installed Packages dotNetRDF, 3.5.2 pythonnet, 3.0.5

dotNetRDF et Python.NET chargés ; RDF, SPARQL et SHACL disponibles.


## Exemple guidé 1 — Proposer une extension

L'état initial décrit cinq classes, quatre propriétés et la disjonction de `Book` et `Member`. L'extension déclare `Ebook`, `format` et un livre numérique. Le diff porte uniquement sur les assertions explicites.

In [2]:
var prefixes = @"@prefix ex: <http://example.org/library#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
";
Graph Lire(string ttl) { var g = new Graph(); StringParser.Parse(g, prefixes + ttl); return g; }
Graph Copier(IGraph original) { var g = new Graph(); g.Merge(original); return g; }
var baseGraph = Lire(@"ex:Book a owl:Class ; owl:disjointWith ex:Member .
ex:Member a owl:Class ; owl:disjointWith ex:Book .
ex:Loan a owl:Class . ex:Library a owl:Class . ex:Person a owl:Class .
ex:title a owl:DatatypeProperty ; rdfs:domain ex:Book ; rdfs:range xsd:string .
ex:author a owl:DatatypeProperty ; rdfs:domain ex:Book ; rdfs:range xsd:string .
ex:name a owl:DatatypeProperty ; rdfs:domain ex:Member ; rdfs:range xsd:string .
ex:borrows a owl:ObjectProperty ; rdfs:domain ex:Member ; rdfs:range ex:Book .");
var extension = Lire(@"ex:Ebook a owl:Class ; rdfs:subClassOf ex:Book .
ex:format a owl:DatatypeProperty ; rdfs:domain ex:Ebook ; rdfs:range xsd:string .
ex:ebook1 a ex:Ebook ; ex:format ""PDF"" .");
var extended = Copier(baseGraph); extended.Merge(extension);
var added = extended.Triples.Except(baseGraph.Triples).ToList();
Console.WriteLine($"Base : {baseGraph.Triples.Count} triplets ; extension : {added.Count} triplets.");
foreach (var t in added.OrderBy(t => t.ToString())) Console.WriteLine(t);

Base : 19 triplets ; extension : 7 triplets.


http://example.org/library#Ebook , http://www.w3.org/1999/02/22-rdf-syntax-ns#type , http://www.w3.org/2002/07/owl#Class


http://example.org/library#Ebook , http://www.w3.org/2000/01/rdf-schema#subClassOf , http://example.org/library#Book


http://example.org/library#ebook1 , http://example.org/library#format , PDF^^http://www.w3.org/2001/XMLSchema#string


http://example.org/library#ebook1 , http://www.w3.org/1999/02/22-rdf-syntax-ns#type , http://example.org/library#Ebook


http://example.org/library#format , http://www.w3.org/1999/02/22-rdf-syntax-ns#type , http://www.w3.org/2002/07/owl#DatatypeProperty


http://example.org/library#format , http://www.w3.org/2000/01/rdf-schema#domain , http://example.org/library#Ebook


http://example.org/library#format , http://www.w3.org/2000/01/rdf-schema#range , http://www.w3.org/2001/XMLSchema#string


Les sept assertions composent la déclaration de classe, la propriété et l'instance. Le diff reste distinct des conséquences calculées plus loin.

### Exercice 1 — Extension concurrente

Déclarez `Audiobook` sous-classe de `Book`, disjointe de `Ebook`, puis `duration` de domaine `Audiobook` et de portée `xsd:integer`. Ajoutez `audio1` et affichez les assertions nouvelles. **Indice** : copiez le graphe initial, puis utilisez `Merge` et `Except`. Contrôlez que le diff ne contient pas les inférences.

In [3]:
Graph audio = null; // TODO : construire l'extension concurrente
if (audio == null) Console.WriteLine("Exercice 1 à compléter : construire le graphe Audiobook.");
else Console.WriteLine($"Diff explicite : {audio.Triples.Except(baseGraph.Triples).Count()} triplets");

Exercice 1 à compléter : construire le graphe Audiobook.


## Exemple guidé 2 — Admissibilité SHACL

Le shape impose au moins un `format` à chaque `Ebook`. Comparons un cas conforme à un cas où `ebook2` n'a pas de format. Le rapport doit identifier le nœud, le chemin et la contrainte.

In [4]:
var shapesGraph = Lire(@"ex:EbookShape a sh:NodeShape ; sh:targetClass ex:Ebook ;
 sh:property ex:EbookShape_format .
ex:EbookShape_format sh:path ex:format ; sh:minCount 1 .");
var shapes = new ShapesGraph(shapesGraph);
var rejected = Copier(baseGraph); rejected.Merge(Lire(@"ex:Ebook a owl:Class ; rdfs:subClassOf ex:Book .
ex:format a owl:DatatypeProperty ; rdfs:domain ex:Ebook ; rdfs:range xsd:string .
ex:ebook2 a ex:Ebook ."));
void Rapport(string nom, IGraph data) {
 var report = shapes.Validate(data);
 Console.WriteLine($"{nom} : conforme={report.Conforms}, résultats={report.Results.Count()}");
 foreach (var r in report.Results) Console.WriteLine($"  focus={r.FocusNode}, chemin={r.ResultPath}, contrainte={r.SourceConstraintComponent}");
}
Rapport("Extension", extended); Rapport("Rejetée", rejected);

Extension : conforme=True, résultats=0


Rejetée : conforme=False, résultats=1


  focus=http://example.org/library#ebook2, chemin=http://example.org/library#format, contrainte=http://www.w3.org/ns/shacl#MinCountConstraintComponent


Le cas rejeté vérifie que la contrainte n'est pas seulement déclarée : elle identifie `ebook2`, le chemin `format` et `MinCountConstraintComponent`.

### Exercice 2 — Deux formes indépendantes

Ajoutez une forme pour `Audiobook` imposant `duration` entière, et une contrainte de type chaîne pour `Ebook`. Validez séparément un livre et un livre audio sans leur propriété ; vérifiez que chaque rapport nomme la bonne forme. **Indice** : deux `sh:targetClass` distincts et deux `sh:property`.

In [5]:
IGraph formesDeuxClasses = null; // TODO : définir les deux formes
if (formesDeuxClasses == null) Console.WriteLine("Exercice 2 à compléter : définir les formes et comparer les rapports.");
else Console.WriteLine($"Formes créées : {formesDeuxClasses.Triples.Count} triplets");

Exercice 2 à compléter : définir les formes et comparer les rapports.


## Exemple guidé 3 — Fermeture OWL-RL et coup nul

Nous sérialisons chaque graphe C# en N-Triples et appelons `owlrl.DeductiveClosure(OWLRL_Semantics)` dans Python.NET. Les ensembles de lignes retournées permettent de comparer toutes les conséquences. Le delta entre fermetures inclut aussi les sept assertions de l'extension : ce n'est donc pas seulement l'ensemble des conséquences non explicites.

In [6]:
if (!PythonEngine.IsInitialized) {
 Runtime.PythonDLL = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL") ?? throw new Exception("Définir PYTHONNET_PYDLL avant le démarrage du kernel.");
 PythonEngine.Initialize();
 PythonEngine.BeginAllowThreads();
}
using (Py.GIL()) {
 dynamic sys = Py.Import("sys");
 var site = Environment.GetEnvironmentVariable("OWL_RL_SITE_PACKAGES");
 if (!String.IsNullOrEmpty(site)) sys.path.insert(0, site);
 dynamic owl = Py.Import("owlrl"); dynamic rdf = Py.Import("rdflib");
 Console.WriteLine($"Moteur owlrl {owl.__version__} ; rdflib {rdf.__version__}");
}
HashSet<string> Fermer(IGraph data) {
 var nt = VDS.RDF.Writing.StringWriter.Write(data, new NTriplesWriter());
 using (Py.GIL()) {
  dynamic rdf = Py.Import("rdflib"); dynamic owl = Py.Import("owlrl");
  dynamic graph = rdf.Graph(); graph.parse(data: nt, format: "nt");
  owl.DeductiveClosure(owl.OWLRL_Semantics).expand(graph);
  string result = graph.serialize(format: "nt").ToString();
  return result.Split('\n', StringSplitOptions.RemoveEmptyEntries).ToHashSet();
 }
}
var baseClosed = Fermer(baseGraph); var extClosed = Fermer(extended);
var delta = extClosed.Except(baseClosed).OrderBy(s => s).ToList();
Console.WriteLine($"Fermeture initiale : {baseClosed.Count} ; étendue : {extClosed.Count} ; delta : {delta.Count}");
foreach (var t in delta) Console.WriteLine(t);
var nul = Copier(baseGraph); nul.Merge(Lire("ex:Book rdfs:subClassOf owl:Thing ."));
var nullDelta = Fermer(nul).Except(baseClosed).Count();
Console.WriteLine($"Coup nul : {nul.Triples.Count - baseGraph.Triples.Count} assertion ajoutée, delta de fermeture = {nullDelta}");
Graph GrapheValide(IEnumerable<string> lignes) {
 var g = new Graph();
 StringParser.Parse(g, String.Join("\n", lignes.Where(l => !l.StartsWith("\""))), new NTriplesParser());
 return g;
}
Rapport("Extension après OWL-RL", GrapheValide(extClosed));
Rapport("Rejetée après OWL-RL", GrapheValide(Fermer(rejected)));

Moteur owlrl 7.1.4 ; rdflib 7.6.0


Fermeture initiale : 174 ; étendue : 196 ; delta : 22


"PDF"^^<http://www.w3.org/2001/XMLSchema#string> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2001/XMLSchema#string> .


"PDF"^^<http://www.w3.org/2001/XMLSchema#string> <http://www.w3.org/2002/07/owl#sameAs> "PDF"^^<http://www.w3.org/2001/XMLSchema#string> .


<http://example.org/library#Ebook> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#Class> .


<http://example.org/library#Ebook> <http://www.w3.org/2000/01/rdf-schema#subClassOf> <http://example.org/library#Book> .


... (18 outputs left out)

Coup nul : 1 assertion ajoutée, delta de fermeture = 0


Extension après OWL-RL : conforme=True, résultats=0


Rejetée après OWL-RL : conforme=False, résultats=1


  focus=http://example.org/library#ebook2, chemin=http://example.org/library#format, contrainte=http://www.w3.org/ns/shacl#MinCountConstraintComponent


Le delta contient notamment `ebook1 rdf:type Book`, conséquence de la sous-classe. La validation après fermeture conserve les verdicts conforme et rejeté ; pour ce contrôle SHACL, seules les lignes RDF à sujet valide sont réimportées. Le coup nul affirme un fait déjà présent dans la fermeture : il ajoute une assertion explicite sans ajouter de conséquence. `owlrl` produit aussi des lignes dont le sujet est un littéral ; elles sont conservées ici comme texte pour comparer fidèlement les fermetures, mais ne sont pas réimportées comme triples RDF 1.1 dans dotNetRDF.

### Exercice 3 — Comparer deux extensions

Comparez l'extension complète à une variante qui conserve `Ebook`, `format` et `ebook1`, mais omet `Ebook rdfs:subClassOf Book`. Calculez les deux deltas et cherchez le type inféré de `ebook1`. **Indice** : utilisez `Fermer` sur deux copies ; ne mutez pas la base.

In [7]:
IGraph sansSousClasse = null; // TODO : construire la variante puis comparer les fermetures
if (sansSousClasse == null) Console.WriteLine("Exercice 3 à compléter : comparer les deux deltas OWL-RL.");
else Console.WriteLine($"Delta variante : {Fermer(sansSousClasse).Except(baseClosed).Count()}");

Exercice 3 à compléter : comparer les deux deltas OWL-RL.


## Exemple guidé 4 — Réification et provenance

Une ressource `rdf:Statement` identifie les trois composantes de l'assertion, avec auteur et date PROV. C'est de la **réification RDF 1.1**, pas une triple citée RDF-star : la réification seule n'affirme pas le triplet, qui figure déjà dans le graphe étendu.

In [8]:
var provenance = Copier(extended);
provenance.Merge(Lire(@"_:assertion a rdf:Statement ; rdf:subject ex:ebook1 ; rdf:predicate rdf:type ;
 rdf:object ex:Ebook ; prov:wasAttributedTo ex:alice ;
 prov:generatedAtTime ""2026-08-22T00:00:00""^^xsd:dateTime ."));
Console.WriteLine($"Avant : {extended.Triples.Count} ; après provenance : {provenance.Triples.Count} triplets");
var query = @"PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX ex: <http://example.org/library#>
SELECT ?s ?p ?o ?date WHERE { ?a a rdf:Statement ; rdf:subject ?s ; rdf:predicate ?p ; rdf:object ?o ; prov:wasAttributedTo ex:alice ; prov:generatedAtTime ?date . }";
var rows = (SparqlResultSet)provenance.ExecuteQuery(query);
foreach (var row in rows) Console.WriteLine($"Alice : {row["s"]} {row["p"]} {row["o"]}, date {row["date"]}");
var turtle = VDS.RDF.Writing.StringWriter.Write(provenance, new CompressingTurtleWriter());
var roundtrip = new Graph(); StringParser.Parse(roundtrip, turtle);
Console.WriteLine($"Sérialisation Turtle puis relecture : {roundtrip.Triples.Count} triplets ; isomorphe = {provenance.Equals(roundtrip)}");

Avant : 26 ; après provenance : 32 triplets


Alice : http://example.org/library#ebook1 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://example.org/library#Ebook, date 2026-08-22T00:00:00^^http://www.w3.org/2001/XMLSchema#dateTime


Sérialisation Turtle puis relecture : 32 triplets ; isomorphe = True


La requête retrouve l'assertion attribuée à Alice et sa date ; la relecture vérifie que la sérialisation préserve le graphe, y compris le nœud blanc.

## À retenir

Le diff explicite, la validation SHACL, le delta OWL-RL et la provenance répondent à quatre questions complémentaires. OWL-RL reste un profil d'OWL 2, pas un raisonneur OWL-DL complet ; SHACL contrôle les contraintes données, sans prouver une cohérence logique générale.

## Vérifiez votre compréhension

1. Pourquoi une assertion nouvelle peut-elle laisser la fermeture inchangée ?
2. Quel élément du rapport SHACL permet de localiser une violation ?
3. Pourquoi la réification n'affirme-t-elle pas à elle seule le triplet décrit ?

<details><summary>Éléments de réponse</summary>

1. Elle peut être déjà déduite ; 2. le focus et le chemin ; 3. elle décrit une ressource représentant l'assertion, distincte de l'assertion elle-même.
</details>

## Pour aller plus loin

Reprenez [SW-8 SHACL](SW-08-SHACL-CSharp.ipynb), [SW-13 raisonneurs](SW-13-Reasoners-CSharp.ipynb), puis comparez avec le [carnet Python](SW-14-Coup-Ontologique-Python.ipynb).